In [1]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()  # 加载.env文件里的变量
# print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

In [2]:
from typing import Optional, Union

from openai import BaseModel
from pydantic import Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate


# llm.invoke("你好")

class UserInfo(BaseModel):
    """Extracted user information, such as name,age, phone,email"""
    name:str=Field(description="The name of user")
    age:Optional[int]=Field(description="The age of user")
    email:str=Field(description="The email of user")
    phone:Optional[str]=Field(description="The phone of user")

class ConversationalResponse(BaseModel):
    response:str=Field(description="chat response from LLM")    

class FinalResponse(BaseModel):
    final_output:Union[UserInfo,ConversationalResponse]
    

parser=PydanticOutputParser(pydantic_object=FinalResponse)
prompt=ChatPromptTemplate.from_messages([
    ('system','解析用户输入并提取个人信息 {format_instructions}'),
    ('human','{query}')
])

prompt=prompt.partial(format_instructions=parser.get_format_instructions())
structured_llm=prompt | llm | parser



In [3]:
structured_llm.invoke("我叫奥特曼，今年38岁，邮箱地址是aoteman@qq.com,电话是123123123")

FinalResponse(final_output=UserInfo(name='奥特曼', age=38, email='aoteman@qq.com', phone='123123123'))

In [4]:
from langchain_core.tools import tool


@tool
def fetch_real_time_info(query: str):
    """fetch real time info"""
    print('fetch_real_time_info', query)
    return {'message': '天气晴'}

In [5]:
from langgraph.prebuilt import ToolNode

tools=[fetch_real_time_info]
tool_node=ToolNode(tools)

In [6]:
from langchain_core.messages import AIMessage
from langgraph._internal._constants import CONF, CONFIG_KEY_RUNTIME
from langgraph.runtime import Runtime

message_with_single_tool_call=AIMessage(
    content="",
    tool_calls=[{
        'name':'fetch_real_time_info',
        'args':{'query':'小米汽车'},
        'id':'tool_call_id',
        'type':'tool_call'
    }]
)


config={CONF:{CONFIG_KEY_RUNTIME:Runtime()}}
tool_node.invoke({"messages":[message_with_single_tool_call]},config)

fetch_real_time_info 小米汽车


{'messages': [ToolMessage(content='{"message": "天气晴"}', name='fetch_real_time_info', tool_call_id='tool_call_id')]}

In [7]:
@tool
def get_weather(location):
    """call to get the current weather."""
    if location.lower() in ["beijing"]:
        return "北京的温度是16度，天气晴朗。"
    elif location.lower() in ["shanghai"]:
        return "上海的温度是30度，天气多云"
    else:
        return "不好意思，并未查询到具体的天气信息"

In [8]:
tools=[fetch_real_time_info,get_weather]
tool_node=ToolNode(tools)

In [9]:
message_with_multiple_tool_calls=AIMessage(
    content="",
    tool_calls=[
        {
            "name":"fetch_real_time_info",
        "args":{"query":"小米汽车"},
        "id":"tool_call_id",
        "type":"tool_call"
        },
        {
            "name":"get_weather",
            "args":{"location":"beijing"},
            "id":"tool_call_id1",
            "type":"tool_call"
        }
    ]
)

config = {CONF: {CONFIG_KEY_RUNTIME: Runtime()}}
tool_node.invoke({"messages":[message_with_multiple_tool_calls]},config)

fetch_real_time_info 小米汽车


{'messages': [ToolMessage(content='{"message": "天气晴"}', name='fetch_real_time_info', tool_call_id='tool_call_id'),
  ToolMessage(content='北京的温度是16度，天气晴朗。', name='get_weather', tool_call_id='tool_call_id1')]}

In [10]:
model_with_tools=llm.bind_tools(tools)

In [11]:
model_with_tools.kwargs

{'tools': [{'type': 'function',
   'function': {'name': 'fetch_real_time_info',
    'description': 'fetch real time info',
    'parameters': {'properties': {'query': {'type': 'string'}},
     'required': ['query'],
     'type': 'object'}}},
  {'type': 'function',
   'function': {'name': 'get_weather',
    'description': 'call to get the current weather.',
    'parameters': {'properties': {'location': {}},
     'required': ['location'],
     'type': 'object'}}}]}

In [12]:
model_with_tools.invoke("beijing的天气").tool_calls

[{'name': 'get_weather',
  'args': {'location': '北京'},
  'id': 'call_00_fu3ZNXQ3yVVL7BcoWjGA7989',
  'type': 'tool_call'}]

In [30]:
tool_node.invoke({"messages":[model_with_tools.invoke("小米汽车最新消息")]},config)

fetch_real_time_info 小米汽车最新消息


{'messages': [ToolMessage(content='{"message": "天气晴"}', name='fetch_real_time_info', tool_call_id='call_00_dqTtDk8Zlyfx3sZydLsQ8804')]}